# 05 — Contrôle pré-prod (garde-fou AVANT le build HTML)

Vérifie que `data/donnees.csv` (format pivot long) respecte le **contrat**
(`docs/contrat_donnees_pivot.md`) et les **modalités attendues**
(`docs/descriptif_sources.yaml`) AVANT de lancer le build des ~500 pages.

S'exécute à l'identique sur données **fictives** (générateur Option B) et **réelles**
(chargeur YAML) — il ne lit que `donnees.csv` et les vues larges via `load_*`.

1. Chargement (long + vues larges).
2. Statistiques de couverture.
3. Tests ✓/✗ (les tests *critiques* font échouer le notebook → bloquent le build).
4. Exploration libre.

## 1. Chargement

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from pathlib import Path
import pandas as pd, numpy as np, yaml

DATA_DIR = Path('../data')
from report_builder import load_aphp, load_survival, load_regional, load_delais_hopitaux

long = pd.read_csv(DATA_DIR / 'donnees.csv', low_memory=False)
aphp   = load_aphp(DATA_DIR)        # vue large DIM APHP (aphp/ghu)
surv   = load_survival(DATA_DIR)    # vue large survie
reg    = load_regional(DATA_DIR)    # vue large régional
delais = load_delais_hopitaux(DATA_DIR)  # délais aphp/ghu/hôpital

desc = yaml.safe_load(open('../docs/descriptif_sources.yaml', encoding='utf-8'))
ATT = desc['modalites_attendues']
GHU_ATT, HOP_ATT, TYP_ATT, NIV_ATT = (set(ATT['ghu_codes']), set(ATT['hopitaux']),
                                      set(ATT['type_etab']), set(ATT['niveau_interne']))
val = pd.to_numeric(long['valeur'], errors='coerce')
print(f"donnees.csv : {len(long):,} lignes · {long['variable'].nunique()} variables · "
      f"sources {sorted(long['source'].unique())}")

/Users/remi/GitHub/cancer_network_report/notebooks/../src/report_builder.py:605: DtypeWarning: Columns (0: stade) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / "donnees.csv")


/Users/remi/GitHub/cancer_network_report/notebooks/../src/report_builder.py:605: DtypeWarning: Columns (0: stade) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / "donnees.csv")


/Users/remi/GitHub/cancer_network_report/notebooks/../src/report_builder.py:605: DtypeWarning: Columns (0: stade) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / "donnees.csv")


/Users/remi/GitHub/cancer_network_report/notebooks/../src/report_builder.py:605: DtypeWarning: Columns (0: stade) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / "donnees.csv")


donnees.csv : 174,578 lignes · 12 variables · sources ['BN', 'DIM APHP', 'EDS APHP']


## 2. Couverture

In [2]:
# Matrice source × niveau × variable (nb d'observations)
mat = long.pivot_table(index=['source', 'niveau'], columns='variable',
                       values='valeur', aggfunc='count', fill_value=0)
mat

variable            delai_chirurgie_median  delai_global_median  \
source   niveau                                                   
BN       aphp                            0                    0   
         type_etab                       0                    0   
DIM APHP aphp                          288                  288   
         ghu                          1728                 1728   
         hopital                      9216                 9216   
EDS APHP aphp                            0                    0   
         ghu                             0                    0   
         hopital                         0                    0   

variable            delai_radio_median  delai_traitement_medical_median  \
source   niveau                                                           
BN       aphp                        0                                0   
         type_etab                   0                                0   
DIM APHP aphp                      288                              288   
         ghu                      1728                             1728   
         hopital                  9216                             9216   
EDS APHP aphp                        0                                0   
         ghu                         0                                0   
         hopital                     0                                0   

variable            nb_patients  nb_patients_stade  nb_sejours_chimiotherapie  \
source   niveau                                                                 
BN       aphp               576                  0                        288   
         type_etab         2880                  0                       1440   
DIM APHP aphp               576                  0                        288   
         ghu               3456                  0                       1728   
         hopital          18432                  0                       9216   
EDS APHP aphp                 0               1408                          0   
         ghu                  0               8448                          0   
         hopital              0               8192                          0   

variable            nb_sejours_chirurgie  nb_sejours_palliatifs  \
source   niveau                                                   
BN       aphp                        288                    288   
         type_etab                  1440                   1440   
DIM APHP aphp                        288                    288   
         ghu                        1728                   1728   
         hopital                    9216                   9216   
EDS APHP aphp                          0                      0   
         ghu                           0                      0   
         hopital                       0                      0   

variable            nb_sejours_radiotherapie  survie_1an  survie_5ans  
source   niveau                                                        
BN       aphp                            288           0            0  
         type_etab                      1440           0            0  
DIM APHP aphp                            288           0            0  
         ghu                            1728           0            0  
         hopital                        9216           0            0  
EDS APHP aphp                              0        1352         1352  
         ghu                               0        7656         7656  
         hopital                           0        7913         7913

In [3]:
print('Entités par niveau :')
for n in sorted(long['niveau'].unique()):
    print(f"  {n:<10} : {long[long.niveau == n].entite.nunique():>3} entités")
print('\nAnnées par source :')
for s, g in long.groupby('source'):
    print(f"  {s:<9} : {sorted(int(a) for a in g.annee.unique())}")
print('\nValeurs manquantes (NaN) par variable — le long ne stocke PAS de NaN :')
print(f"  valeurs NaN dans donnees.csv : {int(val.isna().sum())}")
# Masquage survie = lignes nb_patients_stade SANS survie correspondante
nbst = len(long[long.variable == 'nb_patients_stade'])
masq = nbst - len(long[long.variable == 'survie_1an'])
print(f"  survie masquée (« — », effectifs < 5) : {masq} / {nbst} = {masq/nbst:.1%}")

Entités par niveau :
  aphp       :   1 entités
  ghu        :   6 entités
  hopital    :  32 entités
  type_etab  :   5 entités

Années par source :
  BN        : [2022, 2023, 2024, 2025]
  DIM APHP  : [2022, 2023, 2024, 2025]
  EDS APHP  : [2022, 2023, 2024, 2025]

Valeurs manquantes (NaN) par variable — le long ne stocke PAS de NaN :
  valeurs NaN dans donnees.csv : 0


  survie masquée (« — », effectifs < 5) : 1127 / 18048 = 6.2%


## 3. Tests ✓/✗

Tests **critiques** (contrat) → font échouer le notebook si rouge (bloquent le build).
Tests **informatifs** (dérive, concordance) → signalés sans bloquer (certains écarts
sont normaux sur les vraies données, ex. noms d'hôpitaux survie ≠ délais).

In [4]:
tests = []
def check(nom, ok, detail='', critique=True):
    tests.append((nom, bool(ok), detail, critique))

# ── Contrat : bornes des mesures ──
sv = val[long.variable.isin(['survie_1an', 'survie_5ans'])]
check('survie ∈ [0,100]', sv.between(0, 100).all(), f'min={sv.min():.0f} max={sv.max():.0f}')
dl = val[long.variable.str.startswith('delai_')]
check('délais ≥ 0', (dl >= 0).all(), f'min={dl.min():.1f}')
ct = val[long.variable.str.startswith('nb_')]
check('comptes ≥ 0', (ct >= 0).all(), f'min={ct.min():.0f}')

# ── Structure / contrat ──
check('3 sources présentes', set(long.source) == {'BN', 'DIM APHP', 'EDS APHP'},
      str(sorted(long.source.unique())))
check('sentinelle TOTAL présente',
      bool(((long.appareil == 'TOTAL') | (long.organe == 'TOTAL')).any()))
check('age = tous partout', set(long.age) == {'tous'}, str(set(long.age)))
check('stade rempli UNIQUEMENT en survie (EDS APHP)',
      set(long[long.stade.notna()].source) <= {'EDS APHP'},
      str(set(long[long.stade.notna()].source)))
nouv = set(long[long.population == 'nouveaux'].variable)
check('population ⊆ {tous,nouveaux} ; « nouveaux » réservé patients/survie',
      set(long.population) <= {'tous', 'nouveaux'}
      and nouv <= {'nb_patients', 'nb_patients_stade', 'survie_1an', 'survie_5ans'},
      f'nouveaux: {sorted(nouv)}')

# ── Dérive (informatif) ──
ghu_obs = set(long[long.niveau == 'ghu'].entite)
check('GHU ⊆ modalités attendues', ghu_obs <= GHU_ATT, str(ghu_obs - GHU_ATT), critique=False)
hop_obs = set(long[long.niveau == 'hopital'].entite)
check('hôpitaux ⊆ modalités attendues', hop_obs <= HOP_ATT,
      str(sorted(hop_obs - HOP_ATT)[:8]), critique=False)
typ_obs = set(long[long.niveau == 'type_etab'].entite)
check('types d\'établissement ⊆ attendus', typ_obs <= TYP_ATT, str(typ_obs - TYP_ATT), critique=False)
check('niveaux ⊆ attendus', set(long.niveau) <= NIV_ATT, str(set(long.niveau) - NIV_ATT), critique=False)

# ── Concordance hôpitaux survie ↔ délais (informatif : noms diffèrent en réel) ──
h_surv = set(long[(long.source == 'EDS APHP') & (long.niveau == 'hopital')].entite)
h_del = set(delais.entite) - {'AP-HP'} - GHU_ATT
recouv = len(h_surv & h_del) / max(1, len(h_surv | h_del))
check('concordance hôpitaux survie ↔ délais', recouv >= 0.8,
      f'{recouv:.0%} de recouvrement ({len(h_surv)} survie / {len(h_del)} délais)', critique=False)

# ── Affichage ──
for nom, ok, detail, crit in tests:
    tag = '✓' if ok else ('✗' if crit else '⚠')
    print(f"  {tag} {nom}" + (f"  ({detail})" if detail else ''))
ko = [t for t in tests if not t[1] and t[3]]
warn = [t for t in tests if not t[1] and not t[3]]
print(f"\n{'— TOUS LES TESTS CRITIQUES SONT VERTS ✓' if not ko else f'— {len(ko)} TEST(S) CRITIQUE(S) ✗'}"
      + (f' · {len(warn)} avertissement(s) ⚠' if warn else ''))
assert not ko, f"Contrôle pré-prod : {len(ko)} test(s) critique(s) en échec → build BLOQUÉ"

  ✓ survie ∈ [0,100]  (min=8 max=99)
  ✓ délais ≥ 0  (min=15.2)
  ✓ comptes ≥ 0  (min=0)
  ✓ 3 sources présentes  (['BN', 'DIM APHP', 'EDS APHP'])
  ✓ sentinelle TOTAL présente
  ✓ age = tous partout  ({'tous'})
  ✓ stade rempli UNIQUEMENT en survie (EDS APHP)  ({'EDS APHP'})
  ✓ population ⊆ {tous,nouveaux} ; « nouveaux » réservé patients/survie  (nouveaux: ['nb_patients', 'nb_patients_stade', 'survie_1an', 'survie_5ans'])
  ✓ GHU ⊆ modalités attendues  (set())
  ✓ hôpitaux ⊆ modalités attendues  ([])
  ✓ types d'établissement ⊆ attendus  (set())
  ✓ niveaux ⊆ attendus  (set())
  ✓ concordance hôpitaux survie ↔ délais  (100% de recouvrement (32 survie / 32 délais))

— TOUS LES TESTS CRITIQUES SONT VERTS ✓


## 4. Exploration libre

Cellule libre pour inspecter le détail (filtrer `long`, comparer les vues larges…).

In [5]:
# Exemple : AP-HP toutes localisations par année (vue large)
aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL') & (aphp.organe == 'TOTAL')][
    ['annee', 'nb_patients', 'nb_nouveaux_patients', 'delai_global_median']
].sort_values('annee').reset_index(drop=True)

,annee,nb_patients,nb_nouveaux_patients,delai_global_median
0,2022,70340,35157,28.268778
1,2023,71636,35841,28.449074
2,2024,77989,38866,27.791000
3,2025,79236,39581,27.325704
